# 10 - Cross-Validation
## Hands-on examples

This notebook demonstrates common cross-validation strategies, scoring, leakage-safe pipelines, model comparison, hyperparameter tuning, group-aware validation, and time-series validation.

**Requirements:** `numpy`, `pandas`, `scikit-learn`, and `matplotlib`.

Install if needed: `python -m pip install numpy pandas scikit-learn matplotlib`


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer, make_classification, make_regression
from sklearn.model_selection import (
    KFold, StratifiedKFold, RepeatedStratifiedKFold, GroupKFold,
    TimeSeriesSplit, cross_val_score, cross_validate, GridSearchCV
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

print("scikit-learn version:", sklearn.__version__)


## 1. Why evaluate across multiple folds?

A single split can be unusually easy or difficult. Cross-validation repeats training/validation using different folds and reveals both average performance and variation.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1")

print("F1 per fold:", np.round(scores, 3))
print(f"Mean F1: {scores.mean():.3f}")
print(f"Standard deviation: {scores.std():.3f}")


## 2. Visualize K-Fold partitions

Every sample appears in one validation fold. Here we visualize which observations belong to the validation fold in each split.

In [ ]:
import matplotlib.pyplot as plt

indices = np.arange(20)
kf = KFold(n_splits=4, shuffle=True, random_state=42)
fold_matrix = np.zeros((4, len(indices)), dtype=int)

for fold, (_, valid_idx) in enumerate(kf.split(indices)):
    fold_matrix[fold, valid_idx] = 1

plt.figure(figsize=(10, 3))
plt.imshow(fold_matrix, aspect="auto", interpolation="nearest")
plt.xlabel("Sample index")
plt.ylabel("Fold")
plt.yticks(range(4), [f"Fold {i+1}" for i in range(4)])
plt.title("Validation samples in 4-Fold CV")
plt.colorbar(label="Validation sample")
plt.show()


## 3. Stratified versus ordinary K-Fold

For classification, `StratifiedKFold` attempts to preserve class proportions in each fold. This is useful when classes are imbalanced.

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=500, n_features=10, n_informative=5, n_redundant=2,
    weights=[0.90, 0.10], random_state=42
)

splitters = {
    "KFold": KFold(n_splits=5, shuffle=True, random_state=42),
    "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
}

for name, splitter in splitters.items():
    print("\\n", name)
    for fold, (_, valid_idx) in enumerate(splitter.split(X_imb, y_imb), start=1):
        counts = np.bincount(y_imb[valid_idx], minlength=2)
        print(f"Fold {fold}: class counts={counts}, positive rate={y_imb[valid_idx].mean():.3f}")


## 4. Multiple metrics with `cross_validate`

`cross_validate` can return several metrics, fit times, and optionally training scores.

In [ ]:
result = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={"accuracy": "accuracy", "precision": "precision",
             "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"},
    return_train_score=True,
    n_jobs=-1,
)

summary = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "f1", "roc_auc"],
    "validation_mean": [
        result[f"test_{m}"].mean()
        for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]
    ],
    "validation_std": [
        result[f"test_{m}"].std()
        for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]
    ],
})
summary.round(3)


## 5. Compare two models on identical folds

Use the same splitter and metric so the comparison is fair. Scaling is included for Logistic Regression; Random Forest does not normally require feature scaling.

In [ ]:
models = {
    "Logistic Regression": model,
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ),
}

rows = []
for name, estimator in models.items():
    fold_scores = cross_val_score(estimator, X, y, cv=cv, scoring="f1", n_jobs=-1)
    rows.append({
        "model": name,
        "fold_scores": np.round(fold_scores, 3).tolist(),
        "mean_f1": fold_scores.mean(),
        "std_f1": fold_scores.std(),
    })

pd.DataFrame(rows).round(3)


## 6. Hyperparameter tuning with GridSearchCV

The scaler belongs inside the pipeline so it is fit only on each training fold.

In [ ]:
search = GridSearchCV(
    estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    param_grid={"logisticregression__C": [0.01, 0.1, 1, 10]},
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
search.fit(X, y)

print("Best C:", search.best_params_["logisticregression__C"])
print("Best mean CV F1:", round(search.best_score_, 3))
pd.DataFrame(search.cv_results_)[
    ["param_logisticregression__C", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")


## 7. Demonstrate leakage-safe preprocessing

Do not fit a scaler on all rows before cross-validation. A pipeline ensures the scaler is learned from each training fold only.

In [ ]:
# Correct pattern
safe_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
safe_scores = cross_val_score(safe_pipeline, X, y, cv=cv, scoring="f1")
print("Leakage-safe F1:", np.round(safe_scores, 3))


## 8. GroupKFold for repeated entities

When multiple records belong to the same customer, patient, or device, group-aware splitting can prevent records from the same entity appearing in both training and validation.

In [ ]:
rng = np.random.default_rng(42)
n_groups, rows_per_group = 12, 4
groups = np.repeat(np.arange(n_groups), rows_per_group)
X_group = rng.normal(size=(len(groups), 5))
y_group = (X_group[:, 0] + rng.normal(size=len(groups)) > 0).astype(int)

group_cv = GroupKFold(n_splits=4)
for fold, (train_idx, valid_idx) in enumerate(
    group_cv.split(X_group, y_group, groups=groups), start=1
):
    train_groups = set(groups[train_idx])
    valid_groups = set(groups[valid_idx])
    assert train_groups.isdisjoint(valid_groups)
    print(f"Fold {fold}: train groups={len(train_groups)}, validation groups={sorted(valid_groups)}")


## 9. TimeSeriesSplit for ordered observations

For forecasting, validation observations should occur after the training observations. `TimeSeriesSplit` grows the training window over time.

In [ ]:
X_time = np.arange(30).reshape(-1, 1)
time_cv = TimeSeriesSplit(n_splits=4)

for fold, (train_idx, valid_idx) in enumerate(time_cv.split(X_time), start=1):
    print(
        f"Fold {fold}: train indices {train_idx[0]}..{train_idx[-1]}, "
        f"validation indices {valid_idx[0]}..{valid_idx[-1]}"
    )
    assert train_idx.max() < valid_idx.min()


## 10. Regression cross-validation

The same concept applies to regression. Negative MAE is used by scikit-learn's scorer convention so larger scorer values are better; negate it to report MAE as a positive loss.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=300, n_features=8, noise=15, random_state=42
)
regression_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
mae_scores = cross_val_score(
    regression_model, X_reg, y_reg, cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error"
)
mae = -mae_scores
print("MAE per fold:", np.round(mae, 2))
print(f"Mean MAE: {mae.mean():.2f} ± {mae.std():.2f}")


## 11. Key takeaways

- Match the splitter to the way data will arrive in production.
- Use `StratifiedKFold` for many classification tasks, `GroupKFold` for grouped data, and `TimeSeriesSplit` for time-ordered data.
- Keep learned preprocessing inside a pipeline.
- Report fold scores plus mean and variability.
- Cross-validation supports model selection; keep a final holdout test set untouched when you need a final unbiased evaluation.

Continue with `practice.md` and check `solution.md` after attempting the exercises.